In [1]:
import os
from pathlib import Path
from typing import Dict, List, Tuple
import re
import markdown
from bs4 import BeautifulSoup

class ReadmeFileValidator:
    def __init__(self, original_path: str, translated_path: str):
        self.original = self._read_file(original_path)
        self.translated = self._read_file(translated_path)
        
    @staticmethod
    def _read_file(file_path: str) -> str:
        """Read content from a file, handling different encodings."""
        encodings = ['utf-8', 'latin-1', 'utf-16']
        
        for encoding in encodings:
            try:
                with open(file_path, 'r', encoding=encoding) as file:
                    return file.read()
            except UnicodeDecodeError:
                continue
        raise ValueError(f"Could not read file {file_path} with any supported encoding")
    
    @staticmethod
    def process_directory(directory: str, output_dir: str = "validation_results") -> Dict[str, dict]:
        """Process all README files in a directory."""
        results = {}
        
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Find all README files
        readme_files = []
        for root, _, files in os.walk(directory):
            for file in files:
                if file.lower().startswith('readme') and file.lower().endswith('.md'):
                    readme_files.append(os.path.join(root, file))
        
        # Process each README pair
        for original_readme in readme_files:
            # Construct translated filename path
            dir_path = os.path.dirname(original_readme)
            base_name = os.path.basename(original_readme)
            translated_name = f"{os.path.splitext(base_name)[0]}_DE.md"
            translated_readme = os.path.join(dir_path, translated_name)
            
            if os.path.exists(translated_readme):
                validator = ReadmeFileValidator(original_readme, translated_readme)
                validation_results = validator.validate_all()
                scoring_results = validator.generate_scores(validation_results)
                
                # Combine validation details + scoring
                combined_results = {
                    "validation": validation_results,
                    "scores": scoring_results
                }
                results[base_name] = combined_results
                
                # Save detailed results
                result_file = os.path.join(output_dir, f"{base_name}_validation.txt")
                with open(result_file, 'w', encoding='utf-8') as f:
                    f.write(f"Validation and Scoring Results for {base_name}\n")
                    f.write("=" * 60 + "\n\n")
                    
                    # Print raw validation results
                    for category, data in validation_results.items():
                        f.write(f"\n{category.upper()} VALIDATION:\n")
                        f.write("-" * 30 + "\n")
                        for key, value in data.items():
                            f.write(f"{key}: {value}\n")

                    # Print scores
                    f.write("\nSCORES:\n")
                    f.write("-" * 30 + "\n")
                    for score_category, score_value in scoring_results.items():
                        f.write(f"{score_category}: {score_value}\n")
        
        return results

    def validate_code_blocks(self) -> Dict[str, bool]:
        """Validate code blocks are preserved with awareness of translated placeholders."""
        def extract_code_blocks(text: str) -> List[str]:
            # Match triple backtick blocks
            code_blocks = re.findall(r'```[\s\S]*?```', text)
            # Optionally match indented code as well
            code_blocks.extend(re.findall(r'(?m)^( {4,}|\t+).*$', text))
            return code_blocks
        
        def normalize_block(block: str) -> str:
            # Remove backticks and language identifier
            cleaned = re.sub(r'^```\w*\n|```$', '', block)
            # Extract just the commands, replacing placeholders with a standard token
            commands = re.sub(r'\[.*?\]', '[PLACEHOLDER]', cleaned)
            return commands.strip()
        
        def blocks_equivalent(orig: str, trans: str) -> bool:
            # Normalize both blocks
            norm_orig = normalize_block(orig)
            norm_trans = normalize_block(trans)
            
            # Compare normalized versions
            if norm_orig == norm_trans:
                # Check if they have the same number of placeholders
                orig_placeholders = len(re.findall(r'\[.*?\]', orig))
                trans_placeholders = len(re.findall(r'\[.*?\]', trans))
                return orig_placeholders == trans_placeholders
            return False
        
        orig_blocks = extract_code_blocks(self.original)
        trans_blocks = extract_code_blocks(self.translated)
        
        # Check if each original block has an equivalent translated block
        content_preserved = True
        if len(orig_blocks) == len(trans_blocks):
            for orig in orig_blocks:
                if not any(blocks_equivalent(orig, trans) for trans in trans_blocks):
                    content_preserved = False
                    break
        else:
            content_preserved = False
    
        return {
            'count_match': len(orig_blocks) == len(trans_blocks),
            'content_preserved': content_preserved,
            'original_count': len(orig_blocks),
            'translated_count': len(trans_blocks)
        }

    def validate_urls(self) -> Dict[str, List[str]]:
        """Validate URLs are preserved and valid."""
        def clean_url(url: str) -> str:
            # Handle nested markdown links
            if '](' in url:
                # Extract the final URL from nested markdown
                nested_match = re.search(r'.*\]\((.*?)\)', url)
                if nested_match:
                    url = nested_match.group(1)
            
            # Remove trailing punctuation and parentheses
            url = re.sub(r'[.,\]\)]+$', '', url)
            # Remove any leading or trailing whitespace
            return url.strip()
    
        def extract_urls(text: str) -> List[str]:
            # Match markdown links: [text](URL)
            urls = re.findall(r'\[.*?\]\((.*?)\)', text)
            
            # Match plain URLs
            urls.extend(re.findall(
                r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|'
                r'(?:%[0-9a-fA-F][0-9a-fA-F]))+', text
            ))
            
            # Clean and normalize URLs
            cleaned_urls = []
            for url in urls:
                cleaned = clean_url(url)
                if cleaned:  # Only add non-empty URLs
                    cleaned_urls.append(cleaned)
                    
            return list(set(cleaned_urls))
        
        orig_urls = extract_urls(self.original)
        trans_urls = extract_urls(self.translated)
        
        # For debugging
        print("Original URLs:", orig_urls)
        print("Translated URLs:", trans_urls)
        
        missing_urls = [url for url in orig_urls if url not in trans_urls]
        extra_urls = [url for url in trans_urls if url not in orig_urls]
        
        return {
            'missing_urls': missing_urls,
            'extra_urls': extra_urls,
            'preserved_count': len(set(orig_urls) & set(trans_urls))
        }

    def validate_markdown_formatting(self) -> Dict[str, Dict[str, int]]:
        """Validate markdown formatting elements are preserved."""
        def count_markdown_elements(text: str) -> Dict[str, int]:
            return {
                'headers': len(re.findall(r'^#{1,6}\s', text, re.MULTILINE)),
                'bold': len(re.findall(r'\*\*.*?\*\*|__.*?__', text)),
                'italic': len(re.findall(r'\*[^*]+\*|_[^_]+_', text)),
                'lists': len(re.findall(r'^\s*[-*+]\s', text, re.MULTILINE)),
                'numbered_lists': len(re.findall(r'^\s*\d+\.\s', text, re.MULTILINE)),
                'tables': len(re.findall(r'\|.*\|', text)),
                'blockquotes': len(re.findall(r'^\s*>', text, re.MULTILINE))
            }
        
        orig_elements = count_markdown_elements(self.original)
        trans_elements = count_markdown_elements(self.translated)
        
        return {
            'original': orig_elements,
            'translated': trans_elements,
            'matches': {
                k: (orig_elements[k] == trans_elements[k]) 
                for k in orig_elements.keys()
            }
        }

    def validate_all(self) -> Dict[str, dict]:
        """Run all validations and return comprehensive results."""
        return {
            'code_blocks': self.validate_code_blocks(),
            'urls': self.validate_urls(),
            'markdown': self.validate_markdown_formatting()
        }

    def generate_scores(self, validation_results: Dict[str, dict]) -> Dict[str, int]:
        """
        Given the raw validation results, produce scores (0–100) for:
          - code preservation
          - URL preservation
          - Markdown formatting preservation
        """
        # --- 1) CODE BLOCKS SCORING ---
        code_data = validation_results['code_blocks']
        code_score = 0
        # For example, if the count matches, give half the points; if the exact content is preserved, give the other half
        if code_data['count_match']:
            code_score += 50
        # content_preserved is True/False for *all* blocks. Adjust logic if partial matches are allowed.
        if code_data['content_preserved']:
            code_score += 50
        
        code_score = min(code_score, 100)

        # --- 2) URLS SCORING ---
        urls_data = validation_results['urls']
        missing_urls = urls_data['missing_urls']
        extra_urls = urls_data['extra_urls']
        preserved_count = urls_data['preserved_count']
        
        # Calculate total original URLs
        original_count = preserved_count + len(missing_urls)
        
        if original_count == 0:
            urls_score = 100  # Perfect score if there were no URLs to preserve
        else:
            # Base score calculation
            # 70% of score based on preservation ratio
            preservation_ratio = preserved_count / original_count
            preservation_score = 70 * preservation_ratio
            
            # 30% of score based on avoiding extra URLs
            extra_urls_penalty = 0
            if extra_urls:
                # Calculate penalty based on ratio of extra to original URLs
                extra_ratio = len(extra_urls) / original_count
                # Cap the penalty at 30 points
                extra_urls_penalty = min(30, 30 * extra_ratio)
            
            urls_score = preservation_score + (30 - extra_urls_penalty)
            
            # Round to nearest integer
            urls_score = round(urls_score)
            
            # Ensure score stays within 0-100 range
            urls_score = max(0, min(urls_score, 100))
        # --- 3) MARKDOWN FORMATTING SCORING ---
        md_data = validation_results['markdown']
        matches_dict = md_data['matches']

        # Let's say we have 7 categories: headers, bold, italic, lists, numbered_lists, tables, blockquotes
        total_categories = len(matches_dict)
        matched_categories = sum(val is True for val in matches_dict.values())

        # Weighted approach: each matched category is worth equal points
        if total_categories > 0:
            md_score = int((matched_categories / total_categories) * 100)
        else:
            md_score = 0

        # --- Combine or just return them individually ---
        return {
            'code_score': code_score,
            'urls_score': urls_score,
            'markdown_score': md_score
        }


if __name__ == "__main__":
    # Example usage with a single README pair
    validator = ReadmeFileValidator(
        "/Users/elijahadejumo/LLM translation/animate/readme_animate-css_animate.css.md",
        "/Users/elijahadejumo/LLM translation/animate/readme_translatedClaude.md"
    )
    single_results = validator.validate_all()
    single_scores = validator.generate_scores(single_results)
    scores_list = list(single_scores.values())  # e.g. [80, 90, 75]
    average_score = sum(scores_list) / len(scores_list)
    file_path =  "/Users/elijahadejumo/LLM translation/animate/readme_animate-css_animate.css.md"

    file_size = os.path.getsize(file_path)
    
  
    
    # Print validation breakdown
    print("\nVALIDATION DETAILS:")
    for category, data in single_results.items():
        print(f"\n{category.upper()} VALIDATION:")
        for key, value in data.items():
            print(f"  {key}: {value}") 

    # Print final scores
    print("\nSCORES:")
    for key, val in single_scores.items():
        print(f"  {key}: {val}")

    print("Average Score:", average_score)
    print(f"File size in bytes: {file_size}")

Original URLs: ['https://avatars2.githubusercontent.com/u/439365?s=460&u=512b4cc5324938ae40bbb8f3b7769d335953cd3a&v=4', 'https://img.shields.io/github/release/animate-css/animate.css.svg?style=for-the-badge', 'mailto:animate@eltonmesquita.com', 'https://github.com/animate-css/animate.css/stargazers', 'https://animate.style/', 'https://github.com/eltonmesquita', 'https://codepen.io', 'https://github.com/animate-css/animate.css/tree/a8d92e585b1b302f7749809c3308d5e381f9cb17', 'https://github.com/daneden', 'https://img.shields.io/badge/license-hippocratic%20license-orange.svg?longCache=true&style=for-the-badge', 'https://img.shields.io/github/forks/animate-css/animate.css.svg?style=for-the-badge', 'https://img.shields.io/github/stars/animate-css/animate.css.svg?style=for-the-badge', 'https://github.com/animate-css/animate.css/blob/main/LICENSE', 'https://avatars2.githubusercontent.com/u/5007208?s=460&u=418401ee605824272e5dcb955fd64ea24546a857&v=4', 'https://github.com/animate-css/animate.c